In [ ]:
import os
import pandas as pd
import ipywidgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, clear_output

from plot_helper import create_dataframe_from_list_of_result_folders

In [ ]:
#RESULTS_BASE_DIR = "../results_2025-05-12"
RESULTS_BASE_DIR = "../results"

## Load Results Dataframe

Select one or more result folders below, load them into a dataframe, then filter configurations.

In [ ]:
results_folders = sorted(
    f for f in os.listdir(RESULTS_BASE_DIR)
    if os.path.isdir(os.path.join(RESULTS_BASE_DIR, f))
)

loaded_df = pd.DataFrame()
active_df = pd.DataFrame()

loader_output = ipywidgets.Output()
status_html = ipywidgets.HTML()
folder_status_label = ipywidgets.HTML()

folder_checkboxes = [
    ipywidgets.Checkbox(
        value=(idx == 0),
        description=name,
        indent=False,
        layout=ipywidgets.Layout(width="auto"),
    )
    for idx, name in enumerate(results_folders)
]

def get_selected_result_folders():
    return [cb.description for cb in folder_checkboxes if cb.value]

def update_folder_status_label():
    selected = len(get_selected_result_folders())
    total = len(folder_checkboxes)
    folder_status_label.value = f"<b>Folders selected:</b> {selected}/{total}"

def update_load_status():
    if loaded_df.empty:
        status_html.value = "<b>No data loaded.</b>"
        return
    status_html.value = (
        f"<b>Loaded rows:</b> {len(loaded_df)} | "
        f"<b>Folders:</b> {len(get_selected_result_folders())}"
    )

def on_folder_checkbox_change(change):
    if change.get("name") == "value":
        update_folder_status_label()

for cb in folder_checkboxes:
    cb.observe(on_folder_checkbox_change, names="value")

def set_all_folder_checkboxes(value):
    for cb in folder_checkboxes:
        cb.unobserve(on_folder_checkbox_change, names="value")
        cb.value = value
        cb.observe(on_folder_checkbox_change, names="value")
    update_folder_status_label()

select_all_folders_button = ipywidgets.Button(description="Select all", button_style="success")
clear_all_folders_button = ipywidgets.Button(description="Clear all", button_style="warning")
select_all_folders_button.on_click(lambda _: set_all_folder_checkboxes(True))
clear_all_folders_button.on_click(lambda _: set_all_folder_checkboxes(False))

folder_grid = ipywidgets.GridBox(
    children=folder_checkboxes,
    layout=ipywidgets.Layout(
        grid_template_columns="repeat(2, minmax(320px, 1fr))",
        grid_gap="4px 16px",
        max_height="280px",
        overflow_y="auto",
        border="1px solid #dddddd",
        padding="8px",
    ),
)

recursive_checkbox = ipywidgets.Checkbox(
    value=True,
    description="Recursive folder scan",
    indent=False,
    layout=ipywidgets.Layout(width="220px"),
)

deduplicate_checkbox = ipywidgets.Checkbox(
    value=True,
    description="Deduplicate runs",
    indent=False,
    layout=ipywidgets.Layout(width="180px"),
)

load_button = ipywidgets.Button(
    description="Load selected folders",
    button_style="success",
    icon="refresh",
)

def load_selected_folders(_=None):
    global loaded_df, active_df
    selected = get_selected_result_folders()

    with loader_output:
        clear_output(wait=True)

        if not selected:
            loaded_df = pd.DataFrame()
            active_df = pd.DataFrame()
            update_load_status()
            print("Select at least one result folder.")
            return

        folder_paths = [os.path.join(RESULTS_BASE_DIR, name) for name in selected]
        df = create_dataframe_from_list_of_result_folders(
            folder_paths,
            recursive=recursive_checkbox.value,
        )

        if df.empty:
            loaded_df = df
            active_df = df
            update_load_status()
            print("No run payloads found in selected folders.")
            return

        if deduplicate_checkbox.value:
            dedup_cols = [
                "config_name",
                "env",
                "run",
                "seed",
                "steps",
                "success",
                "reward",
                "completed_at",
            ]
            dedup_cols = [c for c in dedup_cols if c in df.columns]
            if dedup_cols:
                df = df.drop_duplicates(subset=dedup_cols, keep="first")

        if "completed_at" in df.columns:
            df["completed_at"] = pd.to_datetime(df["completed_at"], errors="coerce")

        sort_cols = [c for c in ["config_name", "env", "run", "seed"] if c in df.columns]
        if sort_cols:
            df = df.sort_values(sort_cols, kind="stable").reset_index(drop=True)

        loaded_df = df
        active_df = df.copy()
        update_load_status()
        print(f"Loaded {len(loaded_df)} rows from {len(selected)} folder(s).")

load_button.on_click(load_selected_folders)
update_folder_status_label()
update_load_status()

display(
    ipywidgets.VBox([
        ipywidgets.HTML("<b>Step 1:</b> Select one or more result folders (children of RESULTS_BASE_DIR)."),
        ipywidgets.HBox([select_all_folders_button, clear_all_folders_button]),
        folder_status_label,
        folder_grid,
        ipywidgets.HBox([recursive_checkbox, deduplicate_checkbox, load_button]),
        status_html,
        loader_output,
    ])
)

In [ ]:
if active_df.empty:
    raise ValueError(
        "No active dataframe to plot. Load one or more folders in the widget above."
    )

In [ ]:
active_df.head()

In [ ]:
#active_df['config_name'].unique()

In [ ]:
cfg_summary_df = (
    active_df.groupby("config_name", dropna=False)
    .agg(
        success_rate=("success", "mean"),
        run_count=("success", "size"),
    )
    .reset_index()
    .sort_values("config_name", kind="stable")
)
cfg_summary_df["success_rate"] = 100.0 * cfg_summary_df["success_rate"]
#cfg_summary_df

The plots below use `active_df` from the loader widget.

## Plot Style Configuration

In [ ]:
# Paper-oriented plotting presets (includes grayscale print-friendly options).
BASE_PAPER_STYLE = {
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "grid.linestyle": "--",
    "axes.facecolor": "white",
    "figure.facecolor": "white",
}

PAPER_STYLE_PRESETS = {
    "Color Balanced": {
        "axes.prop_cycle": plt.cycler(color=["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#E45756"]),
        "image.cmap": "viridis",
    },
    "Color High Contrast": {
        "axes.prop_cycle": plt.cycler(color=["#1B1B1B", "#0072B2", "#D55E00", "#009E73", "#CC79A7"]),
        "image.cmap": "magma",
        "grid.alpha": 0.28,
    },
    "Grayscale Print": {
        "axes.prop_cycle": plt.cycler(color=["#111111", "#555555", "#888888", "#BBBBBB", "#DDDDDD"]),
        "image.cmap": "Greys",
        "grid.alpha": 0.28,
    },
    "Grayscale High Contrast": {
        "axes.prop_cycle": plt.cycler(color=["#000000", "#444444", "#777777", "#AAAAAA", "#CCCCCC"]),
        "image.cmap": "binary",
        "grid.alpha": 0.3,
        "grid.linestyle": ":",
    },
}

def apply_paper_style(style_name):
    plt.style.use("default")
    plt.rcParams.update(BASE_PAPER_STYLE)
    plt.rcParams.update(PAPER_STYLE_PRESETS[style_name])
    style_status.value = f"<b>Active style:</b> {style_name}"

style_dropdown = ipywidgets.Dropdown(
    options=list(PAPER_STYLE_PRESETS.keys()),
    value="Color Balanced",
    description="Plot style",
    layout=ipywidgets.Layout(width="380px"),
)
style_status = ipywidgets.HTML()

def on_style_change(change):
    if change.get("name") == "value":
        apply_paper_style(change["new"] )

style_dropdown.observe(on_style_change, names="value")
apply_paper_style(style_dropdown.value)

display(
    ipywidgets.VBox([
        ipywidgets.HTML("<b>Paper plotting style</b> (choose before generating plots):"),
        ipywidgets.HBox([style_dropdown]),
        style_status,
    ])
)

## PLOT 1 - Aggregate by Any Column

Pick any column except `completed_at`, `seed`, and `run`, then aggregate and plot.

In [ ]:
EXCLUDED_GROUP_COLUMNS = {"completed_at", "seed", "run", "reward"}

def normalize_group_value(value):
    if pd.isna(value):
        return "<missing>"
    return str(value)

def aggregate_by_column(df, group_column, metric):
    if group_column not in df.columns:
        raise ValueError(f"Column '{group_column}' not found in dataframe.")

    temp = df.copy()
    temp["_group_key"] = temp[group_column].map(normalize_group_value)

    if metric == "success_rate":
        grouped = temp.groupby("_group_key", dropna=False)["success"].agg(["mean", "size"]).reset_index()
        grouped["value"] = 100.0 * grouped["mean"]
        grouped["count"] = grouped["size"]
        y_label = "Success Rate (%)"
        title = f"Success Rate by {group_column}"
    elif metric == "reward_mean":
        grouped = temp.groupby("_group_key", dropna=False)["reward"].agg(["mean", "size"]).reset_index()
        grouped["value"] = grouped["mean"]
        grouped["count"] = grouped["size"]
        y_label = "Mean Reward"
        title = f"Mean Reward by {group_column}"
    elif metric == "steps_mean":
        grouped = temp.groupby("_group_key", dropna=False)["steps"].agg(["mean", "size"]).reset_index()
        grouped["value"] = grouped["mean"]
        grouped["count"] = grouped["size"]
        y_label = "Mean Steps"
        title = f"Mean Steps by {group_column}"
    else:
        raise ValueError(f"Unknown metric: {metric}")

    grouped = grouped.sort_values("_group_key", kind="stable").reset_index(drop=True)
    return grouped, title, y_label

def plot_aggregate_by_column(group_column, metric):
    if active_df.empty:
        print("No active dataframe. Load folders and select configs first.")
        return

    grouped, title, y_label = aggregate_by_column(active_df, group_column, metric)
    if grouped.empty:
        print("No data available for this aggregation.")
        return

    fig, ax = plt.subplots(figsize=(11.5, 5.4))
    bars = ax.bar(grouped["_group_key"], grouped["value"])

    for bar, val, cnt in zip(bars, grouped["value"], grouped["count"]):
        if np.isnan(val):
            continue
        if metric == "success_rate":
            label = f"{val:.1f}%\n(n={int(cnt)})"
        else:
            label = f"{val:.3f}\n(n={int(cnt)})"
        ax.annotate(
            label,
            xy=(bar.get_x() + bar.get_width() / 2, val),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    ax.set_title(title)
    ax.set_xlabel(group_column)
    ax.set_ylabel(y_label)
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    if metric == "success_rate":
        ax.set_ylim(0, 115)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

    value_decimals = 2 if metric == "success_rate" else 3
    table_df = grouped[["_group_key", "value", "count"]].copy()
    table_df = table_df.rename(columns={"_group_key": group_column, "value": y_label, "count": "n"})
    table_df[y_label] = table_df[y_label].round(value_decimals)
    display(table_df.reset_index(drop=True))

In [ ]:
aggregate_metric_options = ["success_rate", "reward_mean", "steps_mean"]
aggregate_column_options = [
    c for c in active_df.columns
    if c not in EXCLUDED_GROUP_COLUMNS
]

if not aggregate_column_options:
    print("No eligible columns available for aggregation.")
else:
    ipywidgets.interact(
        plot_aggregate_by_column,
        group_column=ipywidgets.Dropdown(
            options=aggregate_column_options,
            value="config_name" if "config_name" in aggregate_column_options else aggregate_column_options[0],
            description="Group by",
        ),
        metric=ipywidgets.Dropdown(
            options=aggregate_metric_options,
            value="success_rate",
            description="Metric",
        ),
    );

## PLOT 2 - Aggregate by Any Column Pair (Grouped Bars)

Select two columns and a metric to visualize pairwise aggregates as grouped bars.
This is the bar-plot alternative to the heatmap below.

In [ ]:
def aggregate_pair_grouped(df, x_column, group_column, metric):
    if x_column not in df.columns or group_column not in df.columns:
        raise ValueError("Selected columns are not present in the dataframe.")
    if x_column == group_column:
        raise ValueError("Please choose two different columns.")

    temp = df.copy()
    temp["_x"] = temp[x_column].map(normalize_group_value)
    temp["_g"] = temp[group_column].map(normalize_group_value)

    if metric == "success_rate":
        grouped = temp.groupby(["_x", "_g"], dropna=False)["success"].agg(["mean", "size"]).reset_index()
        grouped["value"] = 100.0 * grouped["mean"]
        y_label = "Success Rate (%)"
        title = f"Success Rate by {x_column} and {group_column}"
        y_limits = (0, 105)
        label_fmt = "{:.1f}%"
    elif metric == "reward_mean":
        grouped = temp.groupby(["_x", "_g"], dropna=False)["reward"].agg(["mean", "size"]).reset_index()
        grouped["value"] = grouped["mean"]
        y_label = "Mean Reward"
        title = f"Mean Reward by {x_column} and {group_column}"
        y_limits = None
        label_fmt = "{:.3f}"
    elif metric == "steps_mean":
        grouped = temp.groupby(["_x", "_g"], dropna=False)["steps"].agg(["mean", "size"]).reset_index()
        grouped["value"] = grouped["mean"]
        y_label = "Mean Steps"
        title = f"Mean Steps by {x_column} and {group_column}"
        y_limits = None
        label_fmt = "{:.3f}"
    else:
        raise ValueError(f"Unknown metric: {metric}")

    grouped = grouped.sort_values(["_x", "_g"], kind="stable").reset_index(drop=True)
    return grouped, title, y_label, y_limits, label_fmt

def plot_aggregate_pair_grouped_bars(x_column, group_column, metric, annotate=True):
    if active_df.empty:
        print("No active dataframe. Load folders and select configs first.")
        return
    if x_column == group_column:
        print("Choose two different columns.")
        return

    try:
        grouped, title, y_label, y_limits, label_fmt = aggregate_pair_grouped(
            active_df, x_column, group_column, metric
        )
    except ValueError as exc:
        print(str(exc))
        return

    if grouped.empty:
        print("No data available for this pairwise aggregation.")
        return

    x_levels = sorted(grouped["_x"].dropna().unique().tolist())
    group_levels = sorted(grouped["_g"].dropna().unique().tolist())
    if not x_levels or not group_levels:
        print("No categories available for plotting.")
        return

    x = np.arange(len(x_levels))
    bar_width = min(0.82 / max(len(group_levels), 1), 0.26)
    offset_start = -0.5 * bar_width * (len(group_levels) - 1)

    fig_w = min(max(8.0, 0.38 * len(x_levels) + 4.8), 16.0)
    fig, ax = plt.subplots(figsize=(fig_w, 4.9))

    table = grouped.set_index(["_x", "_g"])
    for idx, g in enumerate(group_levels):
        values = []
        counts = []
        for xv in x_levels:
            if (xv, g) in table.index:
                row = table.loc[(xv, g)]
                if isinstance(row, pd.DataFrame):
                    row = row.iloc[0]
                values.append(float(row["value"]))
                counts.append(int(row["size"]))
            else:
                values.append(np.nan)
                counts.append(0)

        xpos = x + offset_start + idx * bar_width
        bars = ax.bar(
            xpos,
            values,
            width=bar_width,
            label=str(g),
            edgecolor="black",
            linewidth=0.35,
            alpha=0.92,
            zorder=3,
        )

        if annotate:
            for b, val, cnt in zip(bars, values, counts):
                if np.isnan(val):
                    continue
                ax.annotate(
                    f"{label_fmt.format(val)}\n(n={cnt})",
                    xy=(b.get_x() + b.get_width() / 2, val),
                    xytext=(0, 2),
                    textcoords="offset points",
                    ha="center",
                    va="bottom",
                    fontsize=7,
                )

    ax.set_title(title)
    ax.set_xlabel(x_column)
    ax.set_ylabel(y_label)
    ax.set_xticks(x)
    ax.set_xticklabels(x_levels, rotation=25, ha="right")
    if y_limits is not None:
        ax.set_ylim(*y_limits)
    ax.grid(axis="y", linestyle="--", alpha=0.22, zorder=0)
    ax.legend(title=group_column, frameon=False, ncol=1)
    plt.tight_layout()
    plt.show()

    value_decimals = 2 if metric == "success_rate" else 3
    table_df = grouped[["_x", "_g", "value", "size"]].copy()
    table_df = table_df.rename(columns={"_x": x_column, "_g": group_column, "value": y_label, "size": "n"})
    table_df[y_label] = table_df[y_label].round(value_decimals)
    display(table_df.reset_index(drop=True))

pair_bar_column_options = [c for c in active_df.columns if c not in EXCLUDED_GROUP_COLUMNS]
if len(pair_bar_column_options) < 2:
    print("Need at least two eligible columns to build a pairwise grouped bar plot.")
else:
    default_x = "config_name" if "config_name" in pair_bar_column_options else pair_bar_column_options[0]
    default_group = "env" if "env" in pair_bar_column_options and "env" != default_x else pair_bar_column_options[1]

    ipywidgets.interact(
        plot_aggregate_pair_grouped_bars,
        x_column=ipywidgets.Dropdown(
            options=pair_bar_column_options,
            value=default_x,
            description="Column 1",
        ),
        group_column=ipywidgets.Dropdown(
            options=pair_bar_column_options,
            value=default_group,
            description="Column 2",
        ),
        metric=ipywidgets.Dropdown(
            options=aggregate_metric_options,
            value="success_rate",
            description="Metric",
        ),
        annotate=ipywidgets.Checkbox(
            value=True,
            description="Annotate bars",
            indent=False,
        ),
    );

## PLOT 3 - Aggregate by Any Column Pair

Select two columns and a metric to visualize pairwise aggregates as a heatmap.
This is a general version of the previous grouped plots.

In [ ]:
def aggregate_by_column_pair(df, row_column, col_column, metric):
    if row_column not in df.columns or col_column not in df.columns:
        raise ValueError("Selected columns are not present in the dataframe.")
    if row_column == col_column:
        raise ValueError("Please choose two different columns.")

    temp = df.copy()
    temp["_row"] = temp[row_column].map(normalize_group_value)
    temp["_col"] = temp[col_column].map(normalize_group_value)

    if metric == "success_rate":
        grouped = temp.groupby(["_row", "_col"], dropna=False)["success"].agg(["mean", "size"]).reset_index()
        grouped["value"] = 100.0 * grouped["mean"]
        value_label = "Success Rate (%)"
        title = f"Success Rate by ({row_column}, {col_column})"
        vmin, vmax = 0.0, 100.0
    elif metric == "reward_mean":
        grouped = temp.groupby(["_row", "_col"], dropna=False)["reward"].agg(["mean", "size"]).reset_index()
        grouped["value"] = grouped["mean"]
        value_label = "Mean Reward"
        title = f"Mean Reward by ({row_column}, {col_column})"
        vmin, vmax = None, None
    elif metric == "steps_mean":
        grouped = temp.groupby(["_row", "_col"], dropna=False)["steps"].agg(["mean", "size"]).reset_index()
        grouped["value"] = grouped["mean"]
        value_label = "Mean Steps"
        title = f"Mean Steps by ({row_column}, {col_column})"
        vmin, vmax = None, None
    else:
        raise ValueError(f"Unknown metric: {metric}")

    value_pivot = grouped.pivot(index="_row", columns="_col", values="value")
    count_pivot = grouped.pivot(index="_row", columns="_col", values="size")
    value_pivot = value_pivot.sort_index().sort_index(axis=1)
    count_pivot = count_pivot.reindex(index=value_pivot.index, columns=value_pivot.columns)

    return value_pivot, count_pivot, title, value_label, vmin, vmax

def plot_aggregate_by_column_pair(row_column, col_column, metric, annotate=True):
    if active_df.empty:
        print("No active dataframe. Load folders and select configs first.")
        return
    if row_column == col_column:
        print("Choose two different columns.")
        return

    try:
        value_pivot, count_pivot, title, cbar_label, vmin, vmax = aggregate_by_column_pair(
            active_df, row_column, col_column, metric
        )
    except ValueError as exc:
        print(str(exc))
        return

    if value_pivot.empty:
        print("No data available for this pairwise aggregation.")
        return

    n_rows, n_cols = value_pivot.shape
    fig_w = min(max(6.4, 0.7 * n_cols + 2.8), 13.0)
    fig_h = min(max(4.8, 0.55 * n_rows + 2.4), 11.0)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    arr = value_pivot.to_numpy(dtype=float)
    im = ax.imshow(arr, aspect="auto", vmin=vmin, vmax=vmax)

    ax.set_title(title)
    ax.set_xlabel(col_column)
    ax.set_ylabel(row_column)
    ax.set_xticks(np.arange(n_cols))
    ax.set_yticks(np.arange(n_rows))
    ax.set_xticklabels(value_pivot.columns.tolist(), rotation=30, ha="right")
    ax.set_yticklabels(value_pivot.index.tolist())

    if annotate:
        for i in range(n_rows):
            for j in range(n_cols):
                val = arr[i, j]
                cnt = count_pivot.iloc[i, j] if not count_pivot.empty else np.nan
                if np.isnan(val):
                    continue
                if metric == "success_rate":
                    text = f"{val:.1f}%\n(n={int(cnt)})"
                else:
                    text = f"{val:.3f}\n(n={int(cnt)})"
                ax.text(
                    j,
                    i,
                    text,
                    ha="center",
                    va="center",
                    fontsize=7,
                    color="white" if np.isfinite(val) and val > np.nanmean(arr) else "black",
                )

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(cbar_label)

    plt.tight_layout()
    plt.show()

    value_decimals = 2 if metric == "success_rate" else 3
    table_df = value_pivot.copy()
    table_df.index.name = row_column
    table_df.columns.name = col_column
    table_df = table_df.round(value_decimals)
    display(table_df)

pair_column_options = [c for c in active_df.columns if c not in EXCLUDED_GROUP_COLUMNS]
if len(pair_column_options) < 2:
    print("Need at least two eligible columns to build a column-pair aggregate plot.")
else:
    default_row = "config_name" if "config_name" in pair_column_options else pair_column_options[0]
    default_col = "env" if "env" in pair_column_options and "env" != default_row else pair_column_options[1]

    ipywidgets.interact(
        plot_aggregate_by_column_pair,
        row_column=ipywidgets.Dropdown(
            options=pair_column_options,
            value=default_row,
            description="Row",
        ),
        col_column=ipywidgets.Dropdown(
            options=pair_column_options,
            value=default_col,
            description="Column",
        ),
        metric=ipywidgets.Dropdown(
            options=aggregate_metric_options,
            value="success_rate",
            description="Metric",
        ),
        annotate=ipywidgets.Checkbox(
            value=True,
            description="Annotate cells",
            indent=False,
        ),
    );